# Run the world-camera transformation pipeline

Point this at one or more directories of raw sensor chunks, give it somewhere to
write, and it runs Geoff's reconstruction pipeline over the first chunk of each.

The work is done by `sensors_utility.process_raw_recording`, which is the
project's standard entry point for turning a raw recording into processed
chunks. It calls `world_util.world_transformation_pipeline`, the Python
equivalent of MATLAB `reconstructionPipeline.m`, which takes raw sensor counts
through six stages:

1. convert to double
2. linearize the sensor response, marking saturated pixels `Inf`
3. impute the ceiling and floor pixels
4. flat-field correction
5. equalize the RGB channels
6. convert to absolute radiance

The result is a Bayer radiance map, exactly as MATLAB returns.

`process_raw_recording` also processes the minispectrometer chunks when the
recording has them, writing those beside the world output. If the recording has
none, that pass is a harmless no-op.

Note this is a different path from `preprocessing_pipeline.generate_world_videos`,
which builds viewable `W.avi` files using individual stage flags. This notebook
produces calibrated radiance, not video.

> **Kernel.** This needs an environment with `hdf5storage`, which the default
> `python3` (3.13) kernel does not have. Use the **myenv** kernel
> (`/opt/anaconda3/envs/pylids`, Python 3.10) — the same 3.10 environment the
> rest of the preprocessing code runs under. The notebook is already set to it;
> if you see `ModuleNotFoundError: No module named 'hdf5storage'`, the kernel
> has been switched back to the default.

## Set the recordings and output path

`RECORDINGS` maps a label to a raw chunk directory. Add or remove entries to
change what gets processed; everything below loops over whatever is in it.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# INPUT: one entry per recording to process, mapping a label to its raw chunk
# directory. These hold the .npy files a recording writes out, named
# world_<n>.npy alongside world_<n>_metadata.npy. For a FLIC recording that is
# the <subject>/<activity>/GKA/<recording number> directory.
FLIC_RAW_ROOT: Path = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")

RECORDINGS: dict[str, Path] = {
    "indoor": FLIC_RAW_ROOT / "FLIC_2001" / "walkIndoor" / "GKA" / "1",
    "outdoor": FLIC_RAW_ROOT / "FLIC_2001" / "walkOutdoor" / "GKA" / "1",
}

# OUTPUT: each recording gets its own subdirectory beneath this, holding the W
# and M subdirectories process_raw_recording creates. World chunks land in
# <OUTPUT_ROOT>/<label>/W/world_chunk<n>.mat.
OUTPUT_ROOT: Path = Path("/Volumes/FLIC_processing/world_radiance_scratch")

# Set True to regenerate chunks that have already been written.
OVERWRITE_EXISTING: bool = False

# Process the first chunk only. A frame takes about 1.5 seconds, and a chunk is
# already enough to see what a recording looks like, so this keeps the run short
# no matter how long the recordings are. Widen the end value to take more.
CHUNK_RANGES: dict[str, tuple[int, int]] = {"W": (0, 1), "M": (0, 1)}
# ---------------------------------------------------------------------------


# Make world_util and sensors_utility importable.
PROJECT_ROOT: Path = Path("/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis")
SENSOR_UTILITY_DIR: Path = PROJECT_ROOT / "code" / "library" / "sensor_utility"
sys.path.append(str(SENSOR_UTILITY_DIR))

import world_util
import sensors_utility

for recording_name, raw_path in RECORDINGS.items():
    print(f"{recording_name:10s} <- {raw_path}")
print(f"\noutput     -> {OUTPUT_ROOT}")

## Run the pipeline over every recording

`process_raw_recording` reads each chunk, passes the world frames through
`world_transformation_pipeline`, and writes the result as a MATLAB v7.3 file
containing a `data` array of radiance maps and a `metadata` struct with the AGC
settings and timestamps.

Each recording is written into its own subdirectory so their chunks cannot
collide. At one chunk apiece this takes a couple of minutes per recording.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Remember the first world chunk written for each recording so the figure below
# can load it back.
processed_chunks: dict[str, Path] = {}

for recording_name, raw_path in RECORDINGS.items():
    print(f"=== {recording_name} ===")

    # Give each recording its own output subdirectory.
    recording_output: Path = OUTPUT_ROOT / recording_name

    sensors_utility.process_raw_recording(
        str(raw_path),
        str(recording_output),
        overwrite_existing=OVERWRITE_EXISTING,
        verbose=True,
        chunk_ranges=CHUNK_RANGES,
    )

    # process_raw_recording puts the world results in a W subdirectory.
    written: list[Path] = sorted((recording_output / "W").glob("world_chunk*.mat"))
    if not written:
        raise RuntimeError(
            f"No world chunks were written for {recording_name}. Check that "
            f"{raw_path} holds world_<n>.npy files."
        )

    processed_chunks[recording_name] = written[0]
    print(f"  wrote {written[0].name} ({written[0].stat().st_size / 1e6:.1f} MB) "
          f"to {recording_output / 'W'}\n")

## Compare a frame from each recording

The pipeline returns a Bayer radiance map, which is the analysis product: every
value in it was measured or imputed from measurements.

`demosaic_radiance_map_rcd` fills in the two missing colour channels at every
pixel so the frame can be viewed as a normal RGB image. It is the equivalent of
MATLAB `demosaicRadianceMapRCD`, which `reconstructionPipeline.m` does not call
either; in Geoff's code it appears only in `demoProcessingStages.m` and
`runSimulation.m`, both times feeding straight into a plot. Two thirds of its
output is interpolated, so use it for looking, not for measuring.

Each panel is contrast-stretched on its own, because an outdoor scene can be
orders of magnitude brighter than an indoor one and a shared scale would leave
the darker frame black. The median radiance in each title is on the real
absolute scale, so that is what to compare between recordings.

In [ ]:
# hdf5storage writes MATLAB v7.3, which is HDF5, so h5py reads it directly.
# MATLAB stores arrays transposed relative to NumPy, so the on-disk layout is
# (cols, rows, frames) and a single frame needs transposing after it is sliced.
import h5py

FRAME_INDEX: int = 0


def stretch_for_display(values: np.ndarray) -> np.ndarray:
    """Scale finite values into 0-1 using their 1st and 99th percentiles."""
    finite: np.ndarray = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values)
    lower, upper = np.percentile(finite, (1, 99))
    if upper <= lower:
        return np.zeros_like(values)
    return np.clip((values - lower) / (upper - lower), 0, 1)


# One row per recording, Bayer radiance beside the demosaiced RGB view.
fig, axes = plt.subplots(
    len(processed_chunks), 2,
    figsize=(14, 5 * len(processed_chunks)),
    constrained_layout=True,
    squeeze=False,
)

for row, (recording_name, chunk_path) in enumerate(processed_chunks.items()):
    # Read just the one frame this row needs rather than the whole chunk.
    with h5py.File(chunk_path, "r") as processed:
        bayer_radiance: np.ndarray = processed["data"][:, :, FRAME_INDEX].T

    # Ratio-corrected demosaicing, returning (rows, cols, 3).
    rgb_radiance: np.ndarray = world_util.demosaic_radiance_map_rcd(
        bayer_radiance, bayer_pattern="BGGR"
    )

    # Report the median on the absolute radiance scale, since the images
    # themselves are each stretched independently.
    median_radiance: float = float(np.median(bayer_radiance[np.isfinite(bayer_radiance)]))

    axes[row][0].imshow(stretch_for_display(bayer_radiance), cmap="gray")
    axes[row][0].set_title(f"{recording_name}: Bayer radiance (median {median_radiance:.3g})")

    axes[row][1].imshow(stretch_for_display(rgb_radiance))
    axes[row][1].set_title(f"{recording_name}: RGB radiance (RCD demosaiced)")

    for axis in axes[row]:
        axis.axis("off")

fig.suptitle(f"Frame {FRAME_INDEX} of the first chunk", fontsize=15, fontweight="bold")
plt.show()